# Parse EIA Bulk Natural Gas Data

EIA publishes a bulk download for the whole Natural Gas category as one
file: one JSON object per line, one line per series (~18,000 series in
the full file). This avoids the API's `length` cap entirely and gives
full history in one local pass -- no network calls, no pagination.

**Setup:** place the bulk file at `../data/raw/NG.txt` (source:
https://www.eia.gov/opendata/bulkfiles.htm, "Natural Gas" file). It's
gitignored -- too large to commit, and each user should pull their own
current copy.

## Series identified

Found by scanning the file for `series_id`/`name` matches, not guessed:

| Purpose | series_id pattern | Frequency | Coverage |
|---|---|---|---|
| Henry Hub spot | `NG.RNGWHHD.D` | Daily | 1997-01-07 to present |
| Futures contracts 1-4 | `NG.RNGC{1,2,3,4}.D` | Daily | 1993/94 to **2024-04-05 only** |
| Storage, working gas (5 regions + Lower-48 total) | `NG.NW2_EPG0_SWO_R{31,32,33,34,35,48}_BCF.W` | Weekly | 2010-01-01 to present |
| Storage capacity (5 regions, no combined series exists) | `NG.NGM_EPG0_SACW0_R{83,84,85,86,91}_MMCF.M` | Monthly | 2013-01 to present |

**Important:** the futures series stop updating in April 2024 (EIA
discontinued them in this feed) while spot price is current. Don't
expect a full up-to-date futures curve -- only spot is current.

**Working gas vs. capacity** are two different things, both called
"storage" colloquially: working gas is the current inventory (changes
weekly with injections/withdrawals -- this is what the roadmap's Phase 1
calibration needs), capacity is the physical design limit (changes
slowly, no single combined series exists so it's summed here from the 5
regions). Both are built below since it wasn't clear which one was
wanted; a storage-utilization ratio (working gas / capacity) can be
computed later from `storage_working_gas_weekly.csv` and
`storage_capacity_monthly.csv`.

In [ ]:
import json
import os

import pandas as pd

BULK_FILE = "../data/raw/NG.txt"

WANTED = {
    "spot": "NG.RNGWHHD.D",
    "futures_m1": "NG.RNGC1.D",
    "futures_m2": "NG.RNGC2.D",
    "futures_m3": "NG.RNGC3.D",
    "futures_m4": "NG.RNGC4.D",
    "storage_east": "NG.NW2_EPG0_SWO_R31_BCF.W",
    "storage_midwest": "NG.NW2_EPG0_SWO_R32_BCF.W",
    "storage_south_central": "NG.NW2_EPG0_SWO_R33_BCF.W",
    "storage_mountain": "NG.NW2_EPG0_SWO_R34_BCF.W",
    "storage_pacific": "NG.NW2_EPG0_SWO_R35_BCF.W",
    "storage_lower48_total": "NG.NW2_EPG0_SWO_R48_BCF.W",
    "capacity_east": "NG.NGM_EPG0_SACW0_R83_MMCF.M",
    "capacity_south_central": "NG.NGM_EPG0_SACW0_R84_MMCF.M",
    "capacity_midwest": "NG.NGM_EPG0_SACW0_R85_MMCF.M",
    "capacity_mountain": "NG.NGM_EPG0_SACW0_R86_MMCF.M",
    "capacity_pacific": "NG.NGM_EPG0_SACW0_R91_MMCF.M",
}
ID_TO_KEY = {v: k for k, v in WANTED.items()}

In [ ]:
if not os.path.exists(BULK_FILE):
    raise FileNotFoundError(
        f"{BULK_FILE} not found -- download the Natural Gas bulk file from "
        "https://www.eia.gov/opendata/bulkfiles.htm and save it there."
    )

raw = {}
with open(BULK_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # cheap substring pre-check before the more expensive json.loads
        if not any(sid in line for sid in WANTED.values()):
            continue
        obj = json.loads(line)
        sid = obj.get("series_id")
        if sid in ID_TO_KEY:
            raw[ID_TO_KEY[sid]] = obj

missing = set(WANTED) - set(raw)
if missing:
    print(f"WARNING: {len(missing)} series not found in this bulk file: {missing}")
print(f"Loaded {len(raw)}/{len(WANTED)} series")

In [ ]:
def series_to_frame(obj, key):
    freq = obj["f"]
    date_format = {"D": "%Y%m%d", "W": "%Y%m%d", "M": "%Y%m", "A": "%Y"}[freq]
    dates, values = zip(*obj["data"])
    df = pd.DataFrame({"date": pd.to_datetime(dates, format=date_format), key: values})
    return df.sort_values("date").reset_index(drop=True)


def merge_series(keys):
    df = series_to_frame(raw[keys[0]], keys[0])
    for k in keys[1:]:
        df = df.merge(series_to_frame(raw[k], k), on="date", how="outer")
    return df.sort_values("date").reset_index(drop=True)

## Henry Hub spot + futures (daily)

In [ ]:
henry_hub = merge_series(["spot", "futures_m1", "futures_m2", "futures_m3", "futures_m4"])

print(f"{len(henry_hub)} rows, {henry_hub['date'].min().date()} to {henry_hub['date'].max().date()}")
print(f"futures coverage ends {henry_hub.dropna(subset=['futures_m1'])['date'].max().date()} "
      "-- discontinued in this feed after that; spot stays current")
henry_hub.tail()

## Storage: working gas in underground storage (weekly, 5 regions + Lower-48 total)

In [ ]:
storage_keys = [
    "storage_east", "storage_midwest", "storage_south_central",
    "storage_mountain", "storage_pacific", "storage_lower48_total",
]
storage_working_gas = merge_series(storage_keys)

regional_cols = [k for k in storage_keys if k != "storage_lower48_total"]
regional_sum = storage_working_gas[regional_cols].sum(axis=1)
max_diff = (regional_sum - storage_working_gas["storage_lower48_total"]).abs().max()
print(f"{len(storage_working_gas)} rows, {storage_working_gas['date'].min().date()} to {storage_working_gas['date'].max().date()}")
print(f"Sanity check: max |sum(5 regions) - reported Lower-48 total| = {max_diff} Bcf (rounding-level -> consistent)")
storage_working_gas.tail()

## Storage capacity (monthly, 5 regions -- no combined series exists, so summed here)

In [ ]:
capacity_keys = [
    "capacity_east", "capacity_midwest", "capacity_south_central",
    "capacity_mountain", "capacity_pacific",
]
storage_capacity = merge_series(capacity_keys)
storage_capacity["total_capacity_mmcf"] = storage_capacity[capacity_keys].sum(axis=1)
storage_capacity["total_capacity_bcf"] = storage_capacity["total_capacity_mmcf"] / 1000.0

print(f"{len(storage_capacity)} rows, {storage_capacity['date'].min().date()} to {storage_capacity['date'].max().date()}")
storage_capacity.tail()[["date", "total_capacity_mmcf", "total_capacity_bcf"]]

## Save

Saved into per-source subfolders under `data/processed/` rather than one
flat folder, so it's clear at a glance what each file is and room exists
for TTF/JKM/freight data later without renaming anything:

```
data/processed/
  henry_hub/spot_futures_daily.csv
  storage/working_gas_weekly.csv
  storage/capacity_monthly.csv
```

In [ ]:
os.makedirs("../data/processed/henry_hub", exist_ok=True)
os.makedirs("../data/processed/storage", exist_ok=True)
henry_hub.to_csv("../data/processed/henry_hub/spot_futures_daily.csv", index=False)
storage_working_gas.to_csv("../data/processed/storage/working_gas_weekly.csv", index=False)
storage_capacity.to_csv("../data/processed/storage/capacity_monthly.csv", index=False)
print("Saved to ../data/processed/henry_hub/ and ../data/processed/storage/")